# Fase 3. Preparacion de los datos

**Proyecto:** Prediccion de CTR en anuncios digitales  
**Metodologia:** CRISP-DM  
**Entrada:** `data/raw/pcb_dataset_final.tsv`  
**Salida:** particiones Parquet en `data/processed/ctr_phase3/`

## Objetivo

Construir una tabla analitica lista para la fase de modelado: conservar variables disponibles al momento de la impresion, excluir identificadores y datos posteriores al clic, mantener las categorias en forma nativa y separar los registros cronologicamente en entrenamiento, validacion y prueba.

> Ejecuta las celdas en orden. La lectura y escritura se hace con DuckDB para no cargar los 16 millones de registros completos en memoria. El archivo original no se modifica.

## Decisiones de preparacion

| Decision | Tratamiento | Justificacion |
|---|---|---|
| Objetivo | `click` como entero de 8 bits | Etiqueta binaria 0/1 confirmada en la Fase 2. |
| Predictores | `timestamp`, `campaign`, `cat1``cat9` | Informacion temporal y contextual de la impresion; `campaign` y `cat1``cat9` son categoricas aunque sus codigos sean numeros. |
| Identificador de usuario | Excluir `uid` | Evita memorizar usuarios y reduce el riesgo de sobreajuste. |
| Resultados posteriores | Excluir `conversion`, `conversion_timestamp`, `conversion_id`, `attribution`, `click_pos`, `click_nb` y `cpo` | Pueden revelar conversiones, clics o pedidos posteriores al momento de prediccion. |
| Disponibilidad no confirmada | Excluir `cost` y `time_since_last_click` en esta version | Se requiere confirmar que ambos estan disponibles antes o durante cada impresion; es preferible no introducir fuga temporal. |
| Nulos y `-1` | No imputar; verificar predictores y objetivo y detenerse si aparecen | En la Fase 2 no se hallaron nulos y los `-1` estaban en columnas excluidas. No se asigna un significado generico al valor especial. |
| Categoricas | Conservar codigos enteros, sin one-hot ni label-encoding | Se evita multiplicar columnas y se preservan los codigos originales; CatBoost puede recibirlas como categoricas y XGBoost debe convertirlas a `category` al entrenar. |
| Particion | Cronologica 70 % / 15 % / 15 % por `timestamp` | Simula la prediccion de impresiones futuras y evita mezclar el futuro con entrenamiento. Los empates temporales se mantienen en una sola particion. |
| Balanceo | No aplicar sobremuestreo ni submuestreo | Se conserva la distribucion observada; el balanceo, si se necesita, se decide solo con entrenamiento en la fase de modelado. |

## Indice

1. Configuracion y rutas.
2. Definicion del esquema y las variables.
3. Calculo de cortes temporales.
4. Generacion de Parquet por particion.
5. Validaciones, manifiesto y resumen.
6. Instrucciones de uso para el modelado.

In [1]:
from pathlib import Path
import json

import duckdb
import pandas as pd
from IPython.display import display

NOMBRE_ARCHIVO = "pcb_dataset_final.tsv"
RUTAS_CANDIDATAS = [
    Path.cwd() / "data" / "raw" / NOMBRE_ARCHIVO,
    Path.cwd().parent / "data" / "raw" / NOMBRE_ARCHIVO,
]
DATA_PATH = next((p.resolve() for p in RUTAS_CANDIDATAS if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"No se encontro {NOMBRE_ARCHIVO}. Se busco en: "
        + ", ".join(str(p) for p in RUTAS_CANDIDATAS)
    )

PROJECT_ROOT = DATA_PATH.parents[2]
TEMP_DIR = PROJECT_ROOT / "data" / "temp_duckdb"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ctr_phase3"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

conexion = duckdb.connect(config={
    "memory_limit": "2GB",
    "temp_directory": str(TEMP_DIR),
})

print(f"Entrada: {DATA_PATH}")
print(f"Tamano: {DATA_PATH.stat().st_size / (1024 ** 3):.2f} GiB")
print(f"Directorio de salida: {OUTPUT_DIR}")

Entrada: C:\Users\Valentina\Documents\Mineria de datos\proyecto-prediccion-ctr\data\raw\pcb_dataset_final.tsv
Tamano: 2.41 GiB
Directorio de salida: C:\Users\Valentina\Documents\Mineria de datos\proyecto-prediccion-ctr\data\processed\ctr_phase3


In [2]:
TARGET = "click"
FEATURES = ["timestamp", "campaign"] + [f"cat{i}" for i in range(1, 10)]
CATEGORICAL_FEATURES = ["campaign"] + [f"cat{i}" for i in range(1, 10)]
EXCLUDED_REASONS = {
    "uid": "identificador de usuario; riesgo de memorizacion y sobreajuste",
    "conversion": "resultado posterior a la impresion; fuga de informacion potencial",
    "conversion_timestamp": "momento de conversion posterior",
    "conversion_id": "identificador de conversion posterior",
    "attribution": "resultado de atribucion asociado a una conversion",
    "click_pos": "posicion de clic relacionada con el recorrido posterior",
    "click_nb": "conteo de clics relacionado con el recorrido posterior",
    "cpo": "costo por pedido, disponible despues de un resultado de conversion",
    "cost": "disponibilidad en el momento de impresion no confirmada",
    "time_since_last_click": "disponibilidad temporal y calculo no confirmados",
}
SPLIT_RATIOS = {"train": 0.70, "validation": 0.15, "test": 0.15}
SPLITS = list(SPLIT_RATIOS)

if TARGET in FEATURES or len(FEATURES) != len(set(FEATURES)):
    raise ValueError("La lista de predictores es invalida.")

CONFIG = {
    "target": TARGET,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "excluded_reasons": EXCLUDED_REASONS,
    "split_ratios": SPLIT_RATIOS,
    "split_method": "quantile_disc por timestamp, sin dividir empates",
    "parquet_compression": "ZSTD",
}
source_fingerprint = {
    "name": DATA_PATH.name,
    "size_bytes": DATA_PATH.stat().st_size,
    "modified_ns": DATA_PATH.stat().st_mtime_ns,
}

tabla_decisiones = pd.DataFrame([
    {"Variable": nombre, "Decision": motivo}
    for nombre, motivo in EXCLUDED_REASONS.items()
])
display(tabla_decisiones)
print("Predictores conservados:", FEATURES)
print("Categoricas para CatBoost/XGBoost:", CATEGORICAL_FEATURES)

,Variable,Decision
0,uid,identificador de usuario; riesgo de memorizaci...
1,conversion,resultado posterior a la impresion; fuga de in...
2,conversion_timestamp,momento de conversion posterior
3,conversion_id,identificador de conversion posterior
4,attribution,resultado de atribucion asociado a una conversion
5,click_pos,posicion de clic relacionada con el recorrido ...
6,click_nb,conteo de clics relacionado con el recorrido p...
7,cpo,"costo por pedido, disponible despues de un res..."
8,cost,disponibilidad en el momento de impresion no c...
9,time_since_last_click,disponibilidad temporal y calculo no confirmados


Predictores conservados: ['timestamp', 'campaign', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']
Categoricas para CatBoost/XGBoost: ['campaign', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']


## 1. Cortes temporales

Se obtienen los percentiles discretos 70 y 85 del `timestamp` en el dataset completo. Las filas con el mismo `timestamp` quedan siempre en el mismo grupo: entrenamiento hasta el primer corte, validacion despues del primer corte y hasta el segundo, y prueba despues del segundo. Las proporciones efectivas se reportan despues, porque los empates pueden alejarlas ligeramente de 70/15/15.

In [3]:
ruta_sql = str(DATA_PATH).replace("\\", "/").replace("'", "''")
consulta_cortes = f"""
    SELECT
        COUNT(*) AS total_filas,
        quantile_disc(timestamp, {SPLIT_RATIOS['train']}) AS corte_train,
        quantile_disc(timestamp, {SPLIT_RATIOS['train'] + SPLIT_RATIOS['validation']}) AS corte_validation,
        MIN(timestamp) AS timestamp_min,
        MAX(timestamp) AS timestamp_max
    FROM read_csv_auto('{ruta_sql}', delim='\\t', header=true)
"""
total_filas, corte_train, corte_validation, timestamp_min, timestamp_max = (
    conexion.execute(consulta_cortes).fetchone()
)

if total_filas == 0:
    raise ValueError("El dataset de entrada esta vacio.")
if corte_train is None or corte_validation is None or corte_train >= corte_validation:
    raise ValueError("No se pudieron establecer dos cortes temporales distintos.")

print(f"Filas de entrada: {total_filas:,}")
print(f"Rango de timestamp: {timestamp_min:,} a {timestamp_max:,}")
print(f"Entrenamiento: timestamp <= {corte_train:,}")
print(f"Validacion: {corte_train:,} < timestamp <= {corte_validation:,}")
print(f"Prueba: timestamp > {corte_validation:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filas de entrada: 16,468,027
Rango de timestamp: 0 a 2,671,199
Entrenamiento: timestamp <= 1,848,196
Validacion: 1,848,196 < timestamp <= 2,269,328
Prueba: timestamp > 2,269,328


## 2. Creacion del dataset procesado

Se escriben nicamente los predictores acordados y `click` en Parquet con compresion ZSTD. El directorio queda particionado por `split` (por ejemplo, `split=train/`), y la particion no se incluye como predictor dentro del archivo. Si ya existe una salida, solo se reutiliza cuando el manifiesto confirma que corresponde al mismo archivo de entrada y a la misma configuracion; no se sobrescriben datos ajenos.

In [4]:
if OUTPUT_DIR.exists():
    if not MANIFEST_PATH.is_file():
        raise FileExistsError(
            f"Existe {OUTPUT_DIR} pero no tiene manifest.json. "
            "No se sobrescribira; revisa el contenido antes de continuar."
        )
    manifest_existente = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    config_previa = manifest_existente.get("config", {})
    claves_funcionales = (
        "target", "features", "categorical_features", "split_ratios",
        "split_method", "parquet_compression",
    )
    config_incompatible = any(
        config_previa.get(clave) != CONFIG.get(clave)
        for clave in claves_funcionales
    )
    if manifest_existente.get("source") != source_fingerprint or config_incompatible:
        raise FileExistsError(
            "La salida existente fue creada con otro archivo o configuracion. "
            "No se sobrescribira automaticamente."
        )
    for nombre_split in SPLITS:
        if not list((OUTPUT_DIR / f"split={nombre_split}").glob("*.parquet")):
            raise FileNotFoundError(f"Faltan archivos Parquet para split={nombre_split}.")
    REUTILIZAR_SALIDA = True
else:
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    REUTILIZAR_SALIDA = False

if not REUTILIZAR_SALIDA:
    output_sql = str(OUTPUT_DIR).replace("\\", "/").replace("'", "''")
    columnas_sql = [
        "CAST(timestamp AS BIGINT) AS timestamp",
        "CAST(campaign AS INTEGER) AS campaign",
        *[f"CAST(cat{i} AS INTEGER) AS cat{i}" for i in range(1, 10)],
        "CAST(click AS TINYINT) AS click",
    ]
    expresion_split = (
        f"CASE WHEN timestamp <= {int(corte_train)} THEN 'train' "
        f"WHEN timestamp <= {int(corte_validation)} THEN 'validation' "
        "ELSE 'test' END AS split"
    )
    consulta_copia = f"""
        COPY (
            SELECT {', '.join(columnas_sql)}, {expresion_split}
            FROM read_csv_auto('{ruta_sql}', delim='\\t', header=true)
        )
        TO '{output_sql}'
        (FORMAT PARQUET, PARTITION_BY (split), COMPRESSION ZSTD)
    """
    conexion.execute(consulta_copia)
    print("Se generaron las particiones Parquet.")
else:
    print("Se reutiliza la salida verificada; no se reescribieron archivos.")

Se reutiliza la salida verificada; no se reescribieron archivos.


## 3. Validacion del dataset procesado

Se valida el esquema de salida, la ausencia de nulos y sentinelas en predictores/objetivo, la validez de la etiqueta, el numero total de filas y el orden temporal entre particiones. Si una validacion falla, el notebook se detiene con un error en lugar de declarar la preparacion como exitosa.

In [5]:
resumenes = []
for nombre_split in SPLITS:
    patron = str(OUTPUT_DIR / f"split={nombre_split}" / "*.parquet").replace("\\", "/").replace("'", "''")
    tabla = ", ".join(FEATURES + [TARGET])
    condicion_nulos = " OR ".join(f"{col} IS NULL" for col in FEATURES + [TARGET])
    condicion_sentinelas = " OR ".join(f"{col} = -1" for col in FEATURES)
    consulta_resumen = f"""
        SELECT
            COUNT(*) AS filas,
            COUNT(*) FILTER (WHERE click = 0) AS click_0,
            COUNT(*) FILTER (WHERE click = 1) AS click_1,
            COUNT(*) FILTER (WHERE click NOT IN (0, 1)) AS etiquetas_invalidas,
            COUNT(*) FILTER (WHERE {condicion_nulos}) AS filas_con_nulos,
            COUNT(*) FILTER (WHERE {condicion_sentinelas}) AS filas_con_sentinela,
            MIN(timestamp) AS timestamp_min,
            MAX(timestamp) AS timestamp_max
        FROM read_parquet('{patron}')
    """
    filas, click_0, click_1, invalidas, nulos, sentinelas, minimo, maximo = (
        conexion.execute(consulta_resumen).fetchone()
    )
    if filas == 0:
        raise ValueError(f"La particion {nombre_split} esta vacia.")
    if invalidas or nulos or sentinelas:
        raise ValueError(
            f"La particion {nombre_split} no paso la validacion: "
            f"etiquetas invalidas={invalidas}, filas con nulos={nulos}, "
            f"filas con -1 en predictores={sentinelas}."
        )
    resumenes.append({
        "split": nombre_split,
        "filas": int(filas),
        "porcentaje_dataset": float(filas / total_filas * 100),
        "click_0": int(click_0),
        "click_1": int(click_1),
        "click_1_porcentaje": float(click_1 / filas * 100),
        "timestamp_min": int(minimo),
        "timestamp_max": int(maximo),
    })

resumen_splits = pd.DataFrame(resumenes).set_index("split")
if int(resumen_splits["filas"].sum()) != int(total_filas):
    raise AssertionError("El total de filas de las particiones no coincide con el origen.")
if not (
    resumen_splits.loc["train", "timestamp_max"] <= corte_train
    and resumen_splits.loc["validation", "timestamp_min"] > corte_train
    and resumen_splits.loc["validation", "timestamp_max"] <= corte_validation
    and resumen_splits.loc["test", "timestamp_min"] > corte_validation
):
    raise AssertionError("Las particiones no respetan el orden temporal definido.")
if (resumen_splits[["click_0", "click_1"]] == 0).any().any():
    raise AssertionError("Al menos una particion no contiene ambas clases del objetivo.")

display(resumen_splits)
print(f"Total procesado: {int(resumen_splits['filas'].sum()):,} filas")

,filas,porcentaje_dataset,click_0,click_1,click_1_porcentaje,timestamp_min,timestamp_max
split,,,,,,,
train,11527626,70.000043,7397325,4130301,35.829589,0,1848196
validation,2470203,14.999994,1556926,913277,36.971739,1848197,2269328
test,2470198,14.999963,1566213,903985,36.595649,2269329,2671199


Total procesado: 16,468,027 filas


In [6]:
patron_train = str(OUTPUT_DIR / "split=train" / "*.parquet").replace("\\", "/").replace("'", "''")
esquema_procesado = conexion.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{patron_train}')"
).fetchdf()
display(esquema_procesado)

columnas_esperadas = FEATURES + [TARGET, "split"]
if esquema_procesado["column_name"].tolist() != columnas_esperadas:
    raise AssertionError("El esquema Parquet no coincide con predictores, objetivo y particion.")
print("La columna split se deriva de la carpeta Hive y no se incluye como predictor.")

resumen_splits.to_csv(OUTPUT_DIR / "split_summary.csv", index=True, encoding="utf-8")
manifest = {
    "source": source_fingerprint,
    "config": CONFIG,
    "temporal_cutoffs": {
        "train_end_inclusive": int(corte_train),
        "validation_end_inclusive": int(corte_validation),
        "timestamp_min": int(timestamp_min),
        "timestamp_max": int(timestamp_max),
    },
    "split_summary": resumen_splits.reset_index().to_dict(orient="records"),
    "output_format": "Parquet partitioned by split, ZSTD compression",
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Resumen: {OUTPUT_DIR / 'split_summary.csv'}")
print(f"Manifiesto: {MANIFEST_PATH}")
print("Todas las validaciones de Fase 3 finalizaron correctamente.")

,column_name,column_type,null,key,default,extra
0,timestamp,BIGINT,YES,None,None,None
1,campaign,INTEGER,YES,None,None,None
2,cat1,INTEGER,YES,None,None,None
3,cat2,INTEGER,YES,None,None,None
4,cat3,INTEGER,YES,None,None,None
5,cat4,INTEGER,YES,None,None,None
6,cat5,INTEGER,YES,None,None,None
7,cat6,INTEGER,YES,None,None,None
8,cat7,INTEGER,YES,None,None,None
9,cat8,INTEGER,YES,None,None,None


La columna split se deriva de la carpeta Hive y no se incluye como predictor.
Resumen: C:\Users\Valentina\Documents\Mineria de datos\proyecto-prediccion-ctr\data\processed\ctr_phase3\split_summary.csv
Manifiesto: C:\Users\Valentina\Documents\Mineria de datos\proyecto-prediccion-ctr\data\processed\ctr_phase3\manifest.json
Todas las validaciones de Fase 3 finalizaron correctamente.


## 4. Uso posterior en modelado

Las rutas de las particiones son:

- `data/processed/ctr_phase3/split=train/*.parquet`
- `data/processed/ctr_phase3/split=validation/*.parquet`
- `data/processed/ctr_phase3/split=test/*.parquet`

Consulta filas o carga lotes desde DuckDB para controlar memoria. Para CatBoost, declara `campaign` y `cat1``cat9` como caracteristicas categoricas. Para XGBoost, al leer cada lote con pandas convierte esas columnas a `category` y configura el modelo con soporte categorico (`enable_categorical=True`, `tree_method="hist"`). Ajusta el modelo y el umbral con entrenamiento/validacion; reserva prueba para la evaluacion final.

No se balancearon las particiones ni se codificaron arbitrariamente las categorias. Las probabilidades y metricas se evaluan en una distribucion natural del dataset.

In [7]:
for nombre_split in SPLITS:
    archivos = sorted((OUTPUT_DIR / f"split={nombre_split}").glob("*.parquet"))
    print(f"{nombre_split}: {len(archivos)} archivo(s)")
    for archivo in archivos:
        print(f"  {archivo.relative_to(PROJECT_ROOT)} ({archivo.stat().st_size / (1024 ** 2):.1f} MiB)")
print(f"\nDataset procesado: {sum(p.stat().st_size for p in OUTPUT_DIR.rglob('*.parquet')) / (1024 ** 3):.2f} GiB")
conexion.close()

train: 1 archivo(s)
  data\processed\ctr_phase3\split=train\data_0.parquet (93.3 MiB)
validation: 1 archivo(s)
  data\processed\ctr_phase3\split=validation\data_0.parquet (20.0 MiB)
test: 1 archivo(s)
  data\processed\ctr_phase3\split=test\data_0.parquet (19.8 MiB)

Dataset procesado: 0.13 GiB
